# Contextual Bandit & Off-Policy Evaluation
**Capital One -- Resiliency Intelligence Team**

This notebook answers four questions a portfolio manager would ask:

1. **Why can't we just use supervised learning** to pick the best debt-resolution offer?
2. **How does LinUCB learn** which offer works best -- and for which customer profile?
3. **How do we evaluate a new policy** without running a live A/B test on real customers?
4. **What is the business impact** of switching from a rule-based policy to LinUCB?

---
> **Prerequisites** -- run the training scripts first:
> ```bash
> python scripts/train.py --n-samples 5000
> python scripts/train_bandit.py
> ```

In [ ]:
import sys
sys.path.insert(0, '..')   # allow importing resiliency from project root

import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from resiliency.models.linucb    import LinUCBAgent, LinUCBArm, ARM_LABELS, N_ARMS
from resiliency.models.rl_agent  import extract_rl_state
from resiliency.models.classifier import DefaultRiskClassifier
from resiliency.evaluation import OPEEvaluator, compute_ips_weights, effective_sample_size
from resiliency.data.generator import LABEL_COL

# Import helpers from the training script to avoid duplicating logic
from scripts.train_bandit import (
    compute_bandit_reward, rule_based_action,
    linucb_propensity_vector, _RidgeRewardModel, _RULE_P_CHOSEN,
)

MODELS_DIR = Path('../models')

for p in [MODELS_DIR / 'linucb_agent.pkl',
          MODELS_DIR / 'customers.parquet',
          MODELS_DIR / 'default_risk_classifier.pkl']:
    status = 'OK' if p.exists() else 'MISSING -- run train.py + train_bandit.py first'
    print(f'  {status:<50s}  {p.name}')

pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline
print('Libraries loaded')

---
## Section 1 -- Why Not Supervised Learning?

A supervised model answers *'will this customer default?'*
But treatment selection asks a different question:

> *Given what we know about this customer, which of the four offers will most likely
> resolve the debt -- and how do we learn that without identical offers for everyone first?*

### The core problem: we only observe one outcome per customer

If we offer customer #1 a Payment Plan and they resolve, we never know whether a
Settlement Offer would have worked better. A supervised model trained on historical
offer-outcome pairs inherits the **selection bias** of whatever policy collected that
data -- it cannot generalise to counterfactual actions.

### The exploration-exploitation tradeoff

A bandit algorithm explicitly manages the tension between:
- **Exploitation** -- send the offer that currently looks best
- **Exploration** -- occasionally try other offers to discover if they are better

The cell below runs a toy 4-arm bandit where the *true* best arm (arm 2, mean=0.80)
is not the one a greedy policy would initially discover.

In [ ]:
# ---- Toy 4-arm bandit: exploration vs exploitation
# True arm means (unknown to the agent) -- arm 2 is best
TRUE_MEANS = np.array([0.55, 0.45, 0.80, 0.60])  # arm 2 is best
N_STEPS    = 600

def run_bandit(strategy, n_steps=N_STEPS, seed=7):
    rng    = np.random.default_rng(seed)
    K      = len(TRUE_MEANS)
    counts = np.zeros(K)
    q_hat  = np.zeros(K)
    rewards = []
    for t in range(1, n_steps + 1):
        if strategy == 'greedy':
            arm = (t - 1) if t <= K else int(np.argmax(q_hat))
        elif strategy == 'eps_greedy':
            arm = int(rng.integers(K)) if rng.random() < 0.15 else int(np.argmax(q_hat))
        else:  # ucb
            if t <= K:
                arm = t - 1
            else:
                ucb = q_hat + np.sqrt(2 * np.log(t) / (counts + 1e-9))
                arm = int(np.argmax(ucb))
        r = float(rng.normal(TRUE_MEANS[arm], 0.2))
        counts[arm] += 1
        q_hat[arm]  += (r - q_hat[arm]) / counts[arm]
        rewards.append(r)
    return np.array(rewards), counts

strategies = {
    'Greedy (no exploration)':  'greedy',
    'Eps-Greedy (eps=15%)':     'eps_greedy',
    'UCB1 (LinUCB ancestor)':   'ucb',
}
palette = {
    'Greedy (no exploration)': '#D03027',
    'Eps-Greedy (eps=15%)':    '#F5A623',
    'UCB1 (LinUCB ancestor)':  '#004977',
}
optimal = TRUE_MEANS.max()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for label, strat in strategies.items():
    rewards, _ = run_bandit(strat)
    cumulative  = np.cumsum(rewards)
    regret      = optimal * np.arange(1, N_STEPS + 1) - cumulative
    axes[0].plot(cumulative, label=label, lw=2, color=palette[label])
    axes[1].plot(regret,     label=label, lw=2, color=palette[label])

axes[0].set_title('Cumulative Reward over Time', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Step (customer interaction)')
axes[0].set_ylabel('Cumulative reward')
axes[0].legend(fontsize=10)

axes[1].set_title('Cumulative Regret (lower = better)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Regret vs optimal arm')
axes[1].legend(fontsize=10)

fig.suptitle(
    f'Toy 4-Arm Bandit | True arm means: {TRUE_MEANS} | Best arm = arm 2 (mean=0.80)',
    fontsize=11, y=1.01
)
fig.tight_layout()
plt.show()

print('Arm pull counts after 600 steps:')
for label, strat in strategies.items():
    _, counts = run_bandit(strat)
    print(f'  {label:<35s}  arm counts={counts.astype(int)}')

In [ ]:
# ---- How the LinUCB confidence bonus shrinks as an arm is observed more
# UCB score = exploitation (x.T @ theta) + alpha * exploration (sqrt(x.T @ A_inv @ x))
# The exploration term decreases as A grows with each observation
from resiliency.models.linucb import _ArmState

x_unit      = np.array([1.0])   # 1-D context for clarity
n_obs_range = np.arange(0, 101)
alphas      = [0.5, 1.0, 2.0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: exploration bonus vs number of observations
for alpha in alphas:
    arm = _ArmState(n_features=1)
    bonuses = []
    for n_obs in n_obs_range:
        bonus = float(np.sqrt(x_unit @ arm.A_inv() @ x_unit))
        bonuses.append(alpha * bonus)
        if n_obs < len(n_obs_range) - 1:
            arm.update(x_unit, reward=1.0)
    axes[0].plot(n_obs_range, bonuses, lw=2, label=f'alpha={alpha}')

axes[0].set_title('Exploration Bonus Shrinks With Data', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of observations on this arm')
axes[0].set_ylabel('alpha * sqrt(x.T @ A_inv @ x)  [exploration bonus]')
axes[0].legend(title='Exploration coefficient')
axes[0].annotate('High uncertainty\n(forced exploration)',
                 xy=(3, 1.8), xytext=(18, 1.65),
                 arrowprops=dict(arrowstyle='->', color='grey'), fontsize=9)
axes[0].annotate('Confident (exploit)',
                 xy=(80, 0.11), xytext=(55, 0.28),
                 arrowprops=dict(arrowstyle='->', color='grey'), fontsize=9)

# Right: UCB = exploit + explore, for different amounts of training data
exploit_range = np.linspace(0, 1, 100)
for n_obs, color, lbl in [
    (5,  '#D03027', 'n=5 obs  (large bonus)'),
    (20, '#F5A623', 'n=20 obs (medium bonus)'),
    (80, '#004977', 'n=80 obs (small bonus)'),
]:
    arm = _ArmState(n_features=1)
    for _ in range(n_obs):
        arm.update(x_unit, reward=0.7)
    bonus = float(np.sqrt(x_unit @ arm.A_inv() @ x_unit))
    axes[1].plot(exploit_range, exploit_range + bonus, lw=2, color=color,
                 label=f'{lbl}  (bonus={bonus:.2f})')
axes[1].plot(exploit_range, exploit_range, 'k--', lw=1.2,
             label='Pure exploitation (no bonus)')
axes[1].set_title('UCB Score = Exploit + Explore', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Exploitation term  (x.T @ theta)')
axes[1].set_ylabel('UCB score  (what the agent maximises)')
axes[1].legend(fontsize=9, title='Training data on this arm')

fig.tight_layout()
plt.show()

---
## Section 2 -- LinUCB in Action

We trained a **Disjoint LinUCB** agent on the synthetic customer portfolio.
Each of the four arms maintains an independent ridge-regression model:

    UCB_a(x) = x.T @ theta_a  +  alpha * sqrt(x.T @ A_a_inv @ x)
               ----- exploit -----    ----------- explore -----------

The four arms correspond to:

| Arm | Offer | Best for |
|-----|-------|----------|
| 0 | Payment Plan | Moderately delinquent, stable income |
| 1 | Settlement 30% | Severe hardship, high default risk |
| 2 | Settlement 50% | Prolonged delinquency, moderate risk |
| 3 | Hardship Program | Requested program, temporary hardship |

In [ ]:
# ---- Load the trained LinUCB agent
agent  = LinUCBAgent.load(MODELS_DIR / 'linucb_agent.pkl')
counts = agent.arm_update_counts()
total  = sum(counts.values())

print(f'Total online updates : {total:,}')
print(f'Alpha (exploration)  : {agent.alpha}')
print(f'Context dimension    : {agent.n_features}')
print()
print('Arm update counts (how many times each arm was chosen during training):')
for name, cnt in counts.items():
    bar = '#' * int(40 * cnt / total)
    print(f'  {name:<22s}  {cnt:>7,}  ({100*cnt/total:>5.1f}%)  {bar}')

In [ ]:
# ---- Learning curve: rolling average reward over online updates
rewards_arr = np.array([r for _, r in agent.reward_history])
window      = max(100, len(rewards_arr) // 30)
rolling_avg = pd.Series(rewards_arr).rolling(window, min_periods=1).mean()
cumulative  = np.cumsum(rewards_arr)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(rewards_arr, alpha=0.15, lw=0.6, color='#004977', label='Step reward')
axes[0].plot(rolling_avg, lw=2,       color='#D03027',
             label=f'Rolling avg (window={window})')
axes[0].axhline(rewards_arr.mean(), ls='--', lw=1.2, color='grey',
                label=f'Overall mean = {rewards_arr.mean():.3f}')
axes[0].set_xlabel('Online update step')
axes[0].set_ylabel('Reward')
axes[0].set_title('LinUCB Online Learning Progress', fontweight='bold')
axes[0].legend(fontsize=10)

axes[1].plot(cumulative, lw=2, color='#004977')
axes[1].fill_between(np.arange(len(cumulative)), cumulative, alpha=0.12, color='#004977')
axes[1].set_xlabel('Online update step')
axes[1].set_ylabel('Cumulative reward')
axes[1].set_title('Cumulative Reward Accumulation', fontweight='bold')

fig.tight_layout()
plt.show()

tail_n = int(0.1 * len(rewards_arr))
print(f'Final 10% of training  -- mean reward : {rewards_arr[-tail_n:].mean():.4f}')
print(f'First 10% of training  -- mean reward : {rewards_arr[:tail_n].mean():.4f}')

In [ ]:
# ---- Load customers and compute default probabilities
df            = pd.read_parquet(MODELS_DIR / 'customers.parquet')
clf           = DefaultRiskClassifier.load(MODELS_DIR / 'default_risk_classifier.pkl')
feature_cols  = [c for c in df.columns if c not in {LABEL_COL, 'hardship_severity'}]
default_probs = clf.predict_proba(df[feature_cols])
n             = len(df)
contexts      = np.stack(
    [extract_rl_state(df.iloc[i]) for i in range(n)], axis=0
).astype(np.float64)

print(f'Dataset : {n:,} customers')
print(f'Default prob mean={default_probs.mean():.1%}  '
      f'high-risk(>60%)={(default_probs > 0.60).mean():.1%}')

In [ ]:
# ---- Which arm does LinUCB prefer per customer segment?
linucb_greedy = np.array([agent.select_action(contexts[i]) for i in range(n)])

risk_tier = np.where(default_probs < 0.30, 'LOW (<30%)',
             np.where(default_probs < 0.60, 'MEDIUM (30-60%)', 'HIGH (>60%)'))
severity  = df['hardship_severity'].astype(int).values

tier_order = ['LOW (<30%)', 'MEDIUM (30-60%)', 'HIGH (>60%)']
sev_labels = {0: '0-None', 1: '1-Moderate', 2: '2-Severe'}
arm_names  = [LinUCBArm(i).name for i in range(N_ARMS)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: arm distribution by risk tier
arm_by_tier = pd.crosstab(
    pd.Categorical(risk_tier, categories=tier_order, ordered=True),
    pd.Categorical([LinUCBArm(a).name for a in linucb_greedy], categories=arm_names),
    normalize='index',
)
arm_by_tier.plot.bar(ax=axes[0], stacked=True, colormap='tab10',
                     edgecolor='white', width=0.55)
axes[0].set_title('LinUCB Arm Selection by Default Risk Tier',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Risk Tier')
axes[0].set_ylabel('Fraction of customers')
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[0].set_xticklabels(tier_order, rotation=0)
axes[0].legend(title='Offer arm', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)

# Right: arm distribution by hardship severity
sev_order   = [sev_labels[k] for k in sorted(sev_labels)]
arm_by_sev  = pd.crosstab(
    pd.Categorical([sev_labels[s] for s in severity],
                   categories=sev_order, ordered=True),
    pd.Categorical([LinUCBArm(a).name for a in linucb_greedy], categories=arm_names),
    normalize='index',
)
arm_by_sev.plot.bar(ax=axes[1], stacked=True, colormap='tab10',
                    edgecolor='white', width=0.55)
axes[1].set_title('LinUCB Arm Selection by Hardship Severity',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Hardship Severity')
axes[1].set_ylabel('Fraction of customers')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[1].set_xticklabels(sev_order, rotation=0)
axes[1].legend(title='Offer arm', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)

fig.tight_layout()
plt.show()

In [ ]:
# ---- UCB score heatmap: arm preference per customer (sorted by default risk)
sample_idx = np.random.default_rng(42).choice(n, size=min(300, n), replace=False)
sample_idx = sample_idx[np.argsort(default_probs[sample_idx])]

ucb_matrix = np.array(
    [agent.get_arm_confidence(contexts[i]) for i in sample_idx]
)  # shape (n_sample, 4)

# Row-normalise so colour shows relative preference, not absolute scale
lo = ucb_matrix.min(axis=1, keepdims=True)
hi = ucb_matrix.max(axis=1, keepdims=True)
ucb_norm = (ucb_matrix - lo) / (hi - lo + 1e-9)

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(ucb_norm.T, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
ax.set_yticks(range(N_ARMS))
ax.set_yticklabels([LinUCBArm(i).name for i in range(N_ARMS)], fontsize=10)
ax.set_xlabel('Customer (sorted by default probability, low to high)', fontsize=11)
ax.set_title('Relative UCB Score -- Which Arm Wins for Each Customer?',
             fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax, label='Relative UCB score (row-normalised)')
fig.tight_layout()
plt.show()

print('Brighter = that arm has the highest UCB score for that customer.')
print('As default risk increases (right), settlement arms become dominant.')

---
## Section 3 -- Off-Policy Evaluation

We cannot simply compare average rewards from a live A/B test without treating
some customers with an untested policy. **Off-Policy Evaluation (OPE)** estimates
the expected reward of any target policy from historical data alone.

### Setup
- **Logging policy (pi_0)**: rule-based heuristic (severity / risk / delinquency -> fixed arm)
- **Historical data**: (context, action, reward, propensity) tuples for every customer
- **Target policies**: Rule-based (baseline), LinUCB, Random

### The three estimators

| Estimator | Formula (plain text) | Strength | Weakness |
|-----------|----------------------|----------|----------|
| DM (Direct Method) | mean of r_hat(x, a_target) | Low variance | Biased if reward model is wrong |
| IPS (Importance Sampling) | mean of w_i * r_i  where  w_i = pi1(a_i) / pi0(a_i) | Unbiased | High variance when weights are large |
| DR (Doubly Robust) | DM + IPS(r_i - r_hat(x_i, a_i)) | Consistent if EITHER model is correct | More complex |

**DR is preferred** because it has a safety net: if the reward model is accurate,
the IPS correction term is small (low variance); if the model is wrong, the IPS
correction removes the bias.

In [ ]:
# ---- Reconstruct the rule-based logging data
log_actions   = np.array(
    [rule_based_action(df.iloc[i], float(default_probs[i])) for i in range(n)],
    dtype=np.int64
)
log_rewards   = np.array(
    [compute_bandit_reward(int(log_actions[i]), df.iloc[i], float(default_probs[i]))
     for i in range(n)]
)
log_propensity = np.full(n, _RULE_P_CHOSEN)

print(f'Logging policy -- mean reward : {log_rewards.mean():.4f}')
print('Action distribution:')
for arm in range(N_ARMS):
    cnt = int((log_actions == arm).sum())
    print(f'  {LinUCBArm(arm).name:<22s}  {cnt:>6,} ({100*cnt/n:>5.1f}%)')

In [ ]:
# ---- Compute target policy propensities and IPS diagnostics
# LinUCB: eps-greedy interpretation of the trained agent
linucb_propensity = linucb_propensity_vector(agent, contexts, log_actions, epsilon=0.10)
# Random: uniform over 4 arms
random_propensity = np.full(n, 1.0 / N_ARMS)

ips_w = compute_ips_weights(log_propensity, linucb_propensity)
ess   = effective_sample_size(log_propensity, linucb_propensity)

print(f'LinUCB importance weights  --  mean={ips_w.mean():.3f}  '
      f'max={ips_w.max():.3f}  min={ips_w.min():.3f}')
print(f'Effective Sample Size      --  ESS={ess:.0f} / {n}  ({100*ess/n:.1f}%)')
print()
print('ESS > 50% indicates the policies overlap enough for reliable OPE estimates.')

In [ ]:
# ---- Fit ridge reward model for DM / DR estimators
reward_model = _RidgeRewardModel(n_arms=N_ARMS)
reward_model.fit(contexts, log_actions, log_rewards)

predicted = reward_model(contexts, log_actions)
ss_res = float(np.sum((log_rewards - predicted) ** 2))
ss_tot = float(np.sum((log_rewards - log_rewards.mean()) ** 2))
r2 = 1.0 - ss_res / ss_tot
print(f'Ridge reward model in-sample R^2 = {r2:.4f}')
print('(Higher R^2 = more accurate DM baseline = lower DR variance)')

In [ ]:
# ---- Run OPEEvaluator.compare_policies()
evaluator = OPEEvaluator(
    contexts=contexts,
    actions=log_actions,
    rewards=log_rewards,
    historical_propensity=log_propensity,
    reward_model=reward_model,
)

rng_nb        = np.random.default_rng(0)
random_greedy = rng_nb.integers(0, N_ARMS, size=n)

ope_table = evaluator.compare_policies(
    policies={
        'Rule-Based (logging)': log_propensity.copy(),
        'LinUCB':               linucb_propensity,
        'Random':               random_propensity,
    },
    clip=10.0,
)

# Add DM with each policy's greedy actions (correct DM = score target policy actions)
ope_table['dm_greedy'] = [
    round(evaluator.direct_method(reward_model, contexts, log_actions),    4),
    round(evaluator.direct_method(reward_model, contexts, linucb_greedy),  4),
    round(evaluator.direct_method(reward_model, contexts, random_greedy),  4),
]

display_cols = ['dm_greedy', 'ips', 'snips', 'dr', 'ess', 'ess_pct', 'w_mean']
rename_map   = {
    'dm_greedy': 'DM',
    'ips':       'IPS',
    'snips':     'SNIPS',
    'dr':        'DR (preferred)',
    'ess':       'ESS',
    'ess_pct':   'ESS %',
    'w_mean':    'w_mean',
}

styled = (
    ope_table[display_cols]
    .rename(columns=rename_map)
    .style
    .format({'ESS %': '{:.1f}%', 'w_mean': '{:.3f}'})
    .background_gradient(cmap='RdYlGn', subset=['DM', 'IPS', 'SNIPS', 'DR (preferred)'])
    .set_caption('Off-Policy Evaluation -- Estimated Expected Reward per Policy')
)
styled

In [ ]:
# ---- Visualise: estimated policy values + ESS reliability diagnostic
policies   = ['Rule-Based (logging)', 'LinUCB', 'Random']
estimators = ['dm_greedy', 'ips', 'dr']
est_labels = ['DM (direct method)', 'IPS', 'DR (doubly robust)']
colors     = ['#94a3b8', '#f59e0b', '#004977']

x     = np.arange(len(policies))
width = 0.25

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: estimated values
ax = axes[0]
for i, (col, lbl, clr) in enumerate(zip(estimators, est_labels, colors)):
    vals = [ope_table.loc[p, col] for p in policies]
    ax.bar(x + (i - 1) * width, vals, width, label=lbl, color=clr,
           edgecolor='white', alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels(['Rule-Based\n(logging)', 'LinUCB', 'Random'], fontsize=10)
ax.set_ylabel('Estimated expected reward', fontsize=11)
ax.set_title('Policy Value by Estimator', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# Right: ESS diagnostic (green > 50%, orange 20-50%, red < 20%)
ax2    = axes[1]
ess_v  = [ope_table.loc[p, 'ess_pct'] for p in policies]
b_clrs = ['#2ecc71' if v > 50 else '#f39c12' if v > 20 else '#e74c3c' for v in ess_v]
bars   = ax2.bar(policies, ess_v, color=b_clrs, edgecolor='white', alpha=0.9)
ax2.axhline(50, ls='--', lw=1.5, color='grey', label='50% reliability threshold')
ax2.set_ylabel('Effective Sample Size (%)', fontsize=11)
ax2.set_title('OPE Reliability -- Effective Sample Size', fontsize=12, fontweight='bold')
ax2.yaxis.set_major_formatter(mticker.PercentFormatter())
ax2.legend(fontsize=10)
ax2.set_xticklabels(['Rule-Based\n(logging)', 'LinUCB', 'Random'], fontsize=10)
for bar, v in zip(bars, ess_v):
    ax2.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.5, f'{v:.0f}%',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

fig.tight_layout()
plt.show()

In [ ]:
# ---- Why DR is most reliable: stability as reward model degrades
# Add increasing noise to the reward model and see how each estimator responds
rng_noise   = np.random.default_rng(1)
noise_range = np.linspace(0, 1.0, 15)
dm_vals, dr_vals = [], []

true_ips = evaluator.ips(log_rewards, log_propensity, linucb_propensity)

for ns in noise_range:
    noisy = (lambda X, a, _ns=ns:
             reward_model(X, a) + rng_noise.normal(0, _ns, len(X)))
    dm_vals.append(evaluator.direct_method(noisy, contexts, linucb_greedy))
    dr_vals.append(evaluator.doubly_robust(
        log_rewards, noisy, contexts, log_actions,
        log_propensity, linucb_propensity,
    ))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(noise_range, dm_vals, lw=2, color='#94a3b8', label='DM (direct method)')
ax.plot(noise_range, dr_vals, lw=2, color='#004977', label='DR (doubly robust)')
ax.axhline(true_ips, ls='--', lw=1.5, color='#D03027',
           label=f'IPS = {true_ips:.4f}  (no reward model)')
ax.fill_between(noise_range, dm_vals, dr_vals, alpha=0.10, color='#004977')
ax.set_xlabel('Reward model noise std  (0 = perfect model, 1 = very noisy)', fontsize=11)
ax.set_ylabel('Estimated policy value for LinUCB', fontsize=11)
ax.set_title(
    'DR Stays Stable as the Reward Model Degrades\n'
    '(DR -> DM when model is accurate; DR -> IPS when model is noisy)',
    fontsize=12, fontweight='bold'
)
ax.legend(fontsize=11)
fig.tight_layout()
plt.show()

print('Key insight: DR interpolates between DM and IPS as model quality changes.')
print('This "doubly robust" property means only ONE of the two models needs to be correct.')

---
## Section 4 -- Key Results & Business Impact

We now translate the DR policy-value estimates into terms a portfolio manager can act on.

### From reward to resolution rate

The reward formula:  `r = 2*res_prob - 1.5*cost + 0.5*sat - 0.5*(1-res_prob)*p_default`

At average cost and satisfaction levels we can invert this to recover an implied
resolution probability, and express lift in percentage points.


In [ ]:
# ---- Business impact calculation
dr_rule   = float(ope_table.loc['Rule-Based (logging)', 'dr'])
dr_linucb = float(ope_table.loc['LinUCB',               'dr'])
dr_random = float(ope_table.loc['Random',               'dr'])

lift_pct = 100.0 * (dr_linucb - dr_rule) / abs(dr_rule)

# Invert reward formula to get implied resolution rate
# r = (2 + 0.5*p_d)*res + (-1.5*avg_cost + 0.5*avg_sat - 0.5*p_d)
p_d      = float(default_probs.mean())
avg_cost = 0.09
avg_sat  = 0.74
intercept = -1.5 * avg_cost + 0.5 * avg_sat - 0.5 * p_d
slope     = 2.0 + 0.5 * p_d

res_rule   = (dr_rule   - intercept) / slope
res_linucb = (dr_linucb - intercept) / slope
res_random = (dr_random - intercept) / slope

# Portfolio projection
PORTFOLIO  = 100_000
AVG_BAL    = 4_500
RECOVERY   = 0.65

extra_res  = int((res_linucb - res_rule) * PORTFOLIO)
dollar_lft = extra_res * AVG_BAL * RECOVERY

print('=' * 62)
print('  OFF-POLICY EVALUATION -- BUSINESS IMPACT SUMMARY')
print('=' * 62)
print(f'  Portfolio size              : {PORTFOLIO:>10,} customers')
print(f'  Average account balance     : {AVG_BAL:>10,.0f} USD')
print()
print(f'  DR estimate -- Rule-Based   : {dr_rule:>10.4f}')
print(f'  DR estimate -- LinUCB       : {dr_linucb:>10.4f}  ({lift_pct:+.1f}% vs rule)')
print(f'  DR estimate -- Random       : {dr_random:>10.4f}')
print()
print(f'  Implied resolution rate')
print(f'     Rule-Based               : {res_rule:>9.1%}')
print(f'     LinUCB                   : {res_linucb:>9.1%}   <- {lift_pct:+.1f}% lift')
print(f'     Random                   : {res_random:>9.1%}')
print()
print(f'  Additional resolved accounts (LinUCB vs Rule-Based)')
print(f'     Per {PORTFOLIO:,} customers      : {extra_res:>+10,}')
print(f'     Estimated recovery       : ${dollar_lft:>10,.0f}')
print('=' * 62)

In [ ]:
# ---- Final summary visualisation
pol_labels = ['Rule-Based\n(baseline)', 'LinUCB\n(proposed)', 'Random']
dr_vals_p  = [dr_rule,   dr_linucb, dr_random]
res_vals_p = [res_rule,  res_linucb, res_random]
bar_colors = ['#94a3b8', '#004977',  '#cbd5e1']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: DR estimate
bars = axes[0].bar(pol_labels, dr_vals_p, color=bar_colors,
                   edgecolor='white', alpha=0.92, width=0.5)
axes[0].set_ylabel('Doubly Robust Estimate', fontsize=11)
axes[0].set_title('Expected Per-Customer Reward\n(Doubly Robust Estimator)',
                  fontsize=12, fontweight='bold')
for bar, val in zip(bars, dr_vals_p):
    delta = val - dr_rule
    lbl   = f'{val:.4f}' if abs(delta) < 1e-9 else f'{val:.4f}\n({delta:+.4f})'
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.002, lbl,
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_ylim(0, max(dr_vals_p) * 1.15)

# Right: implied resolution rate
bars2 = axes[1].bar(pol_labels, res_vals_p, color=bar_colors,
                    edgecolor='white', alpha=0.92, width=0.5)
axes[1].set_ylabel('Implied Resolution Rate', fontsize=11)
axes[1].set_title('Estimated Debt Resolution Rate\n(Derived from DR Estimates)',
                  fontsize=12, fontweight='bold')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
for bar, val in zip(bars2, res_vals_p):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.002, f'{val:.1%}',
                 ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[1].set_ylim(0, max(res_vals_p) * 1.15)

fig.suptitle(
    f'LinUCB improves estimated resolution rate by {lift_pct:+.1f}% over rule-based\n'
    f'{extra_res:+,} additional resolutions per {PORTFOLIO:,} customers'
    f'  (~${dollar_lft/1e6:.1f}M recovered)',
    fontsize=11, fontweight='bold', y=1.03
)
fig.tight_layout()
plt.show()

In [ ]:
# ---- Architecture and methodology summary
summary = pd.DataFrame([
    {'Component': 'Risk Classifier',
     'Method':    'XGBoost + Platt calibration',
     'Output':    'P(default) per customer',
     'Role':      'Feature for bandit state vector'},
    {'Component': 'Bandit Agent',
     'Method':    'Disjoint LinUCB  (alpha=1.0)',
     'Output':    'Best offer arm per context',
     'Role':      'Treatment selection policy'},
    {'Component': 'Logging Policy',
     'Method':    'Deterministic rule hierarchy',
     'Output':    '(action, reward, propensity)',
     'Role':      'Historical data source for OPE'},
    {'Component': 'Reward Model',
     'Method':    'Ridge regression (context + arm one-hot)',
     'Output':    'r_hat(x, a)',
     'Role':      'DM baseline + DR correction term'},
    {'Component': 'OPE Estimator',
     'Method':    'Doubly Robust (DM + IPS correction)',
     'Output':    'Expected reward per policy',
     'Role':      'Policy comparison without A/B test'},
]).set_index('Component')

print('=== Resiliency Intelligence -- System Summary ===')
print(summary.to_string())
print()
print(f'=== Key Findings ===')
print(f'  LinUCB DR estimate  : {dr_linucb:.4f} vs Rule-Based {dr_rule:.4f} ({lift_pct:+.1f}% lift)')
print(f'  ESS                 : {ope_table.loc["LinUCB", "ess_pct"]:.0f}%  -- OPE estimates are statistically reliable')
print(f'  Resolution rate     : {res_linucb:.1%} (LinUCB) vs {res_rule:.1%} (rule-based)')
print(f'  Portfolio impact    : {extra_res:+,} additional resolutions per {PORTFOLIO:,} customers')
print(f'  Estimated recovery  : ~${dollar_lft/1e6:.1f}M additional per {PORTFOLIO:,} customers')
print()
print('=== Next Steps ===')
print('  1. Shadow-deploy LinUCB alongside rule-based to collect live propensities')
print('  2. Run DR-evaluated online experiment to narrow confidence intervals')
print('  3. Tune alpha (exploration) based on live-traffic ESS')
print('  4. Extend to 6-arm action space to match QLearningAgent')